# 貨幣資料收集展示

本筆記本展示如何使用 currency_predictor 模組收集貨幣匯率資料並進行基本的視覺化分析。

## 1. 匯入必要套件

In [ ]:
import sys
import os

# 確保可以匯入專案模組
sys.path.append('../src')

# 匯入資料視覺化套件
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# 匯入專案模組
from currency_predictor.data.collectors import YahooFinanceCollector
from currency_predictor.data.storage import DataStorage

# 🎯 完整的全域中文字型設定
import matplotlib as mpl
from matplotlib.font_manager import FontProperties, fontManager

# 設定中文字型路徑
chinese_font_path = "D:/tools/iansui/Iansui-Regular.ttf"

# 檢查字型檔案是否存在
import os
if os.path.exists(chinese_font_path):
    print(f"✅ 找到字型檔案: {chinese_font_path}")
    
    # 創建字型屬性物件
    chinese_font = FontProperties(fname=chinese_font_path)
    font_name = chinese_font.get_name()
    
    # 方法1: 註冊字型到系統
    fontManager.addfont(chinese_font_path)
    
    # 方法2: 設定字型家族的優先順序 (關鍵!)
    mpl.rcParams['font.family'] = ['sans-serif']
    mpl.rcParams['font.sans-serif'] = [font_name, 'DejaVu Sans', 'Arial Unicode MS', 'SimHei', 'Microsoft YaHei']
    
    # 方法3: 直接設定預設字型
    plt.rcParams['font.family'] = font_name
    
    # 解決負號顯示問題
    mpl.rcParams['axes.unicode_minus'] = False
    
    print(f"✅ 已設定中文字型: {font_name}")
    print("✅ 字型設定完成，所有圖表將自動使用中文字型")
    
else:
    print(f"❌ 找不到字型檔案: {chinese_font_path}")
    print("請檢查字型檔案路徑是否正確")


# 設定圖表樣式
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

plt.rcParams['font.family'] = chinese_font.get_name()
plt.rcParams['axes.unicode_minus'] = False

print("\n所有套件匯入成功！")

## 2. 初始化資料收集器

In [ ]:
# 建立資料收集器
collector = YahooFinanceCollector()
storage = DataStorage()

print(f"支援的貨幣對: {collector.supported_pairs}")
print(f"資料儲存路徑: {storage.base_dir}")  # 修正：使用 base_dir 而不是 base_path

## 3. 檢查目標貨幣對的可用性

In [ ]:
# 測試主要目標貨幣對
target_currencies = ['USDTWD=X', 'TWD=X', 'EURUSD=X']

availability_results = {}

for symbol in target_currencies:
    is_available = collector.check_symbol_availability(symbol)
    availability_results[symbol] = is_available
    status = "✅ 可用" if is_available else "❌ 不可用"
    print(f"{symbol}: {status}")

print("\n可用的貨幣對:")
available_symbols = [symbol for symbol, available in availability_results.items() if available]
print(available_symbols)

## 4. 收集整年的匯率資料

In [ ]:
# 選擇第一個可用的貨幣對進行測試
if available_symbols:
    test_symbol = available_symbols[0]
    print(f"使用貨幣對: {test_symbol}")
    
    # 收集整年的資料
    currency_data_yearly = collector.get_currency_data(
        symbol=test_symbol,
        period="1y",
        interval="1d"
    )
    
    # 同時收集一個月資料用於比較
    currency_data_monthly = collector.get_currency_data(
        symbol=test_symbol,
        period="1mo",
        interval="1d"
    )
    
    if currency_data_yearly is not None:
        print(f"成功收集到整年資料 {len(currency_data_yearly)} 筆")
        print(f"整年資料時間範圍: {currency_data_yearly.index.min()} 到 {currency_data_yearly.index.max()}")
        
    if currency_data_monthly is not None:
        print(f"成功收集到一個月資料 {len(currency_data_monthly)} 筆")
        print(f"一個月資料時間範圍: {currency_data_monthly.index.min()} 到 {currency_data_monthly.index.max()}")
        
        print("\n一個月資料概覽:")
        print(currency_data_monthly.head())
        
        print("\n一個月資料統計:")
        print(currency_data_monthly.describe())
    else:
        print("無法取得資料")
else:
    print("沒有可用的貨幣對")

## 5. 資料視覺化分析

In [ ]:
# 📊 第一張圖：一個月資料視覺化分析
if currency_data_monthly is not None:
    # 建立多個子圖
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    # 現在可以直接使用中文標題，不需要 fontproperties 參數
    fig.suptitle(f'{test_symbol} 一個月匯率分析', fontsize=16, fontweight='bold')
    
    # 1. 收盤價趨勢圖
    axes[0, 0].plot(currency_data_monthly.index, currency_data_monthly['Close'], 
                   linewidth=2, color='blue', alpha=0.8)
    axes[0, 0].set_title('收盤價趨勢')  # 直接使用中文
    axes[0, 0].set_xlabel('日期')
    axes[0, 0].set_ylabel('匯率')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. OHLC 蠟燭圖概念 (用箱型圖表示)
    daily_range = currency_data_monthly['High'] - currency_data_monthly['Low']
    axes[0, 1].fill_between(currency_data_monthly.index, 
                           currency_data_monthly['Low'], currency_data_monthly['High'],
                           alpha=0.3, color='gray', label='日內區間')
    axes[0, 1].plot(currency_data_monthly.index, currency_data_monthly['Open'], 
                   'o', markersize=3, color='green', label='開盤價')
    axes[0, 1].plot(currency_data_monthly.index, currency_data_monthly['Close'], 
                   'o', markersize=3, color='red', label='收盤價')
    axes[0, 1].set_title('開高低收價格範圍')  # 直接使用中文
    axes[0, 1].set_xlabel('日期')
    axes[0, 1].set_ylabel('匯率')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. 交易量圖
    if 'Volume' in currency_data_monthly.columns and currency_data_monthly['Volume'].sum() > 0:
        axes[1, 0].bar(currency_data_monthly.index, currency_data_monthly['Volume'], 
                      alpha=0.7, color='orange')
        axes[1, 0].set_title('交易量')  # 直接使用中文
        axes[1, 0].set_xlabel('日期')
        axes[1, 0].set_ylabel('交易量')
        axes[1, 0].tick_params(axis='x', rotation=45)
    else:
        axes[1, 0].text(0.5, 0.5, '無交易量資料', 
                       horizontalalignment='center', verticalalignment='center',
                       transform=axes[1, 0].transAxes, fontsize=12)
        axes[1, 0].set_title('交易量 (無資料)')  # 直接使用中文
    
    # 4. 每日報酬率分布
    returns = currency_data_monthly['Close'].pct_change().dropna()
    axes[1, 1].hist(returns, bins=20, alpha=0.7, color='purple', edgecolor='black')
    axes[1, 1].axvline(returns.mean(), color='red', linestyle='--', 
                      linewidth=2, label=f'平均報酬率: {returns.mean():.4f}')
    axes[1, 1].set_title('每日報酬率分布')  # 直接使用中文
    axes[1, 1].set_xlabel('報酬率')
    axes[1, 1].set_ylabel('頻率')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("無一個月資料可供視覺化")

In [ ]:
# 📊 第二張圖：整年資料視覺化分析
if currency_data_yearly is not None:
    # 建立多個子圖
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'{test_symbol} 整年匯率分析', fontsize=16, fontweight='bold')
    
    # 1. 收盤價趨勢圖
    axes[0, 0].plot(currency_data_yearly.index, currency_data_yearly['Close'], 
                   linewidth=2, color='darkblue', alpha=0.8)
    axes[0, 0].set_title('收盤價趨勢 (整年)')
    axes[0, 0].set_xlabel('日期')
    axes[0, 0].set_ylabel('匯率')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. 月度平均價格趨勢
    monthly_avg = currency_data_yearly.resample('M')['Close'].mean()
    axes[0, 1].plot(monthly_avg.index, monthly_avg, 
                   linewidth=3, color='green', marker='o', markersize=6, label='月平均價')
    axes[0, 1].fill_between(monthly_avg.index, monthly_avg, alpha=0.3, color='green')
    axes[0, 1].set_title('月度平均價格趨勢')
    axes[0, 1].set_xlabel('月份')
    axes[0, 1].set_ylabel('平均匯率')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. 月度波動度分析
    monthly_volatility = currency_data_yearly.resample('M')['Close'].std()
    axes[1, 0].bar(monthly_volatility.index, monthly_volatility, 
                  alpha=0.7, color='red', width=20)
    axes[1, 0].set_title('月度價格波動度')
    axes[1, 0].set_xlabel('月份')
    axes[1, 0].set_ylabel('價格標準差')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. 整年報酬率分布 vs 滾動平均
    returns_yearly = currency_data_yearly['Close'].pct_change().dropna()
    rolling_mean = currency_data_yearly['Close'].rolling(window=30).mean()
    
    # 在同一個子圖中顯示價格和滾動平均
    ax4_twin = axes[1, 1].twinx()
    
    # 左軸：價格
    axes[1, 1].plot(currency_data_yearly.index, currency_data_yearly['Close'], 
                   color='blue', alpha=0.6, linewidth=1, label='收盤價')
    axes[1, 1].plot(rolling_mean.index, rolling_mean, 
                   color='red', linewidth=2, label='30日滾動平均')
    axes[1, 1].set_ylabel('匯率', color='blue')
    axes[1, 1].set_xlabel('日期')
    axes[1, 1].tick_params(axis='y', labelcolor='blue')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    # 右軸：報酬率的滾動標準差
    rolling_volatility = returns_yearly.rolling(window=30).std() * 100
    ax4_twin.plot(rolling_volatility.index, rolling_volatility, 
                 color='orange', linewidth=2, alpha=0.8, label='30日滾動波動度')
    ax4_twin.set_ylabel('波動度 (%)', color='orange')
    ax4_twin.tick_params(axis='y', labelcolor='orange')
    
    axes[1, 1].set_title('價格與波動度變化')
    axes[1, 1].legend(loc='upper left')
    ax4_twin.legend(loc='upper right')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 顯示整年統計摘要
    print(f"\n📊 {test_symbol} 整年統計摘要:")
    print("=" * 50)
    print(f"資料期間: {currency_data_yearly.index.min().strftime('%Y-%m-%d')} 到 {currency_data_yearly.index.max().strftime('%Y-%m-%d')}")
    print(f"總交易天數: {len(currency_data_yearly)} 天")
    print(f"期間最高價: {currency_data_yearly['High'].max():.4f}")
    print(f"期間最低價: {currency_data_yearly['Low'].min():.4f}")
    print(f"期間平均價: {currency_data_yearly['Close'].mean():.4f}")
    
    yearly_return = (currency_data_yearly['Close'].iloc[-1] / currency_data_yearly['Close'].iloc[0] - 1) * 100
    yearly_volatility = returns_yearly.std() * np.sqrt(252) * 100
    
    print(f"整年報酬率: {yearly_return:+.2f}%")
    print(f"年化波動率: {yearly_volatility:.2f}%")
    print(f"夏普比率 (假設無風險利率=0): {(yearly_return/yearly_volatility if yearly_volatility > 0 else 0):.3f}")
    
    # 月度表現分析
    monthly_returns = currency_data_yearly.resample('M')['Close'].last().pct_change().dropna() * 100
    print(f"\n📈 月度表現分析:")
    print("-" * 30)
    for month, ret in monthly_returns.items():
        print(f"{month.strftime('%Y年%m月')}: {ret:+.2f}%")
        
else:
    print("無整年資料可供視覺化")

## 6. 基本統計分析

In [ ]:
# 📊 詳細統計分析比較 (一個月 vs 整年)
print("📊 統計分析比較")
print("=" * 60)

if currency_data_monthly is not None:
    returns_monthly = currency_data_monthly['Close'].pct_change().dropna()
    
    print("\n📅 一個月資料統計:")
    print("-" * 30)
    print(f"資料筆數: {len(currency_data_monthly)}")
    print(f"期間最高價: {currency_data_monthly['High'].max():.4f}")
    print(f"期間最低價: {currency_data_monthly['Low'].min():.4f}")
    print(f"期間平均價: {currency_data_monthly['Close'].mean():.4f}")
    print(f"價格標準差: {currency_data_monthly['Close'].std():.4f}")
    print(f"平均日報酬率: {returns_monthly.mean():.6f}")
    print(f"報酬率標準差: {returns_monthly.std():.6f}")
    print(f"最大單日漲幅: {returns_monthly.max():.6f}")
    print(f"最大單日跌幅: {returns_monthly.min():.6f}")
    
    # 一個月期間表現
    monthly_change = currency_data_monthly['Close'].iloc[-1] - currency_data_monthly['Close'].iloc[0]
    monthly_change_pct = (currency_data_monthly['Close'].iloc[-1] / currency_data_monthly['Close'].iloc[0] - 1) * 100
    print(f"期間價格變化: {monthly_change:+.4f}")
    print(f"期間變化幅度: {monthly_change_pct:+.2f}%")

if currency_data_yearly is not None:
    returns_yearly = currency_data_yearly['Close'].pct_change().dropna()
    
    print("\n📅 整年資料統計:")
    print("-" * 30)
    print(f"資料筆數: {len(currency_data_yearly)}")
    print(f"期間最高價: {currency_data_yearly['High'].max():.4f}")
    print(f"期間最低價: {currency_data_yearly['Low'].min():.4f}")
    print(f"期間平均價: {currency_data_yearly['Close'].mean():.4f}")
    print(f"價格標準差: {currency_data_yearly['Close'].std():.4f}")
    print(f"平均日報酬率: {returns_yearly.mean():.6f}")
    print(f"報酬率標準差: {returns_yearly.std():.6f}")
    print(f"年化波動率: {returns_yearly.std() * np.sqrt(252) * 100:.2f}%")
    print(f"最大單日漲幅: {returns_yearly.max():.6f}")
    print(f"最大單日跌幅: {returns_yearly.min():.6f}")
    
    # 整年期間表現
    yearly_change = currency_data_yearly['Close'].iloc[-1] - currency_data_yearly['Close'].iloc[0]
    yearly_change_pct = (currency_data_yearly['Close'].iloc[-1] / currency_data_yearly['Close'].iloc[0] - 1) * 100
    print(f"期間價格變化: {yearly_change:+.4f}")
    print(f"期間變化幅度: {yearly_change_pct:+.2f}%")
else:
    print("無資料可供分析")

## 7. 儲存資料到本地

In [ ]:
# 💾 儲存資料到本地 (一個月和整年)
print("💾 開始儲存資料...")

# 儲存一個月資料
if currency_data_monthly is not None:
    success_monthly = storage.save_raw_data(
        data=currency_data_monthly,
        symbol=test_symbol.replace('=X', ''),
        period="1mo"
    )
    
    if success_monthly:
        print(f"✅ 一個月資料已儲存")
    else:
        print("❌ 一個月資料儲存失敗")

# 儲存整年資料
if currency_data_yearly is not None:
    success_yearly = storage.save_raw_data(
        data=currency_data_yearly,
        symbol=test_symbol.replace('=X', ''),
        period="1y"
    )
    
    if success_yearly:
        print(f"✅ 整年資料已儲存")
    else:
        print("❌ 整年資料儲存失敗")

print(f"\n📁 資料儲存位置: {storage.base_dir}")

# 列出已儲存的資料
available_data = storage.list_available_data()
if available_data:
    print("\n📂 已儲存的資料檔案:")
    raw_files = available_data.get('raw_files', [])
    processed_files = available_data.get('processed_files', [])
    
    if raw_files:
        print("原始資料檔案:")
        for file_info in raw_files:
            print(f"  - {file_info}")
            
    if processed_files:
        print("處理後資料檔案:")
        for file_info in processed_files:
            print(f"  - {file_info}")
            
    if not raw_files and not processed_files:
        print("  - 尚無已儲存的檔案")
else:
    print("❌ 無法列出已儲存的檔案")

## 8. 測試多個貨幣對

In [ ]:
# 下載並儲存所有需要的貨幣對資料
print("📥 開始下載並儲存所有需要的貨幣對資料...")

# 定義要下載的貨幣對
target_symbols = ['USDTWD=X', 'EURUSD=X', 'EURTWD=X']
downloaded_data = {}

for symbol in target_symbols:
    print(f"\n正在處理 {symbol}...")
    
    # 檢查可用性
    if collector.check_symbol_availability(symbol):
        print(f"  ✅ {symbol} 可用，正在下載...")
        
        # 下載整年資料
        data = collector.get_currency_data(symbol, period="1y", interval="1d")
        if data is not None and not data.empty:
            print(f"  ✅ 成功下載 {len(data)} 筆資料")
            
            # 儲存到本地
            clean_symbol = symbol.replace('=X', '')
            success = storage.save_raw_data(
                data=data,
                symbol=clean_symbol,
                period="1y"
            )
            
            if success:
                print(f"  💾 資料已儲存到本地")
                downloaded_data[symbol] = data
            else:
                print(f"  ❌ 資料儲存失敗")
        else:
            print(f"  ❌ 無法下載 {symbol} 資料")
    else:
        print(f"  ❌ {symbol} 不可用")

print(f"\n✅ 下載完成！成功下載 {len(downloaded_data)} 個貨幣對")

# 列出已儲存的檔案
print("\n📁 檢查已儲存的資料檔案...")
available_data = storage.list_available_data()
if available_data:
    raw_files = available_data.get('raw_files', [])
    if raw_files:
        print("已儲存的原始資料檔案:")
        for file_info in raw_files:
            print(f"  - {file_info}")

# 📊 從已儲存的 CSV 檔案讀取資料進行視覺化
print("\n📊 從已儲存的 CSV 檔案讀取資料進行比較分析...")

import glob
import os

# 讀取所有已儲存的 CSV 檔案
csv_files = glob.glob(os.path.join(storage.raw_dir, "*.csv"))
comparison_data = {}

for csv_file in csv_files:
    try:
        # 從檔名提取貨幣對名稱
        filename = os.path.basename(csv_file)
        symbol_name = filename.split('_')[0]  # 例如從 'USDTWD_1mo_20250928_153841.csv' 提取 'USDTWD'
        
        # 讀取 CSV 檔案
        df = pd.read_csv(csv_file, index_col=0, parse_dates=True)
        comparison_data[f"{symbol_name}=X"] = df['Close']
        print(f"  ✅ 成功讀取 {symbol_name} 資料，共 {len(df)} 筆")
        
    except Exception as e:
        print(f"  ❌ 讀取 {csv_file} 失敗: {e}")

if comparison_data:
    print(f"\n📈 開始繪製 {len(comparison_data)} 個貨幣對的比較圖表...")
    
    # 建立比較圖表
    fig, axes = plt.subplots(2, 1, figsize=(15, 10))
    
    # 上圖：原始價格比較
    for symbol, prices in comparison_data.items():
        axes[0].plot(prices.index, prices, 
                    linewidth=2, label=symbol, alpha=0.8)
    
    axes[0].set_title('各貨幣對價格比較 (原始價格)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('日期')
    axes[0].set_ylabel('匯率')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')
    
    # 下圖：正規化價格比較
    for symbol, prices in comparison_data.items():
        # 正規化價格 (以第一個價格為基準)
        normalized_prices = prices / prices.iloc[0]
        axes[1].plot(normalized_prices.index, normalized_prices, 
                    linewidth=2, label=symbol, alpha=0.8)
    
    axes[1].set_title('各貨幣對表現比較 (正規化，基準=1.0)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('日期')
    axes[1].set_ylabel('正規化價格 (基準=1.0)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 各貨幣對期間表現統計:")
    print("=" * 60)
    for symbol, prices in comparison_data.items():
        period_return = (prices.iloc[-1] / prices.iloc[0] - 1) * 100
        volatility = prices.pct_change().std() * np.sqrt(252) * 100  # 年化波動率
        
        # 顯示起始和結束價格
        start_price = prices.iloc[0]
        end_price = prices.iloc[-1]
        
        print(f"{symbol}:")
        print(f"  起始價格: {start_price:.4f}")
        print(f"  結束價格: {end_price:.4f}")
        print(f"  報酬率: {period_return:+.2f}%")
        print(f"  年化波動率: {volatility:.1f}%")
        print("-" * 40)
        
else:
    print("❌ 沒有可供比較的資料")

## 總結

本筆記本展示了 currency_predictor 資料收集模組的核心功能，現已更新為支援整年資料分析：

### 🔧 核心功能
1. **初始化收集器** - 建立 YahooFinanceCollector 實例
2. **檢查可用性** - 驗證目標貨幣對是否可從 Yahoo Finance 取得資料
3. **資料收集** - 同時取得一個月和整年的 OHLCV 資料
4. **雙時間軸視覺化** - 分別繪製一個月和整年的分析圖表
5. **比較統計分析** - 對比一個月與整年的市場統計指標
6. **資料儲存** - 將不同時間範圍的資料儲存到本地
7. **多貨幣對比較** - 比較不同貨幣對的整年表現

### 📊 視覺化特色
- **第一張圖**：一個月資料的詳細分析 (收盤價趨勢、OHLC、交易量、報酬率分布)
- **第二張圖**：整年資料的宏觀分析 (長期趨勢、月度平均、波動度、滾動統計)

### 📈 分析深度
- 短期 vs 長期表現比較
- 月度統計摘要
- 年化波動率計算
- 滾動平均與波動度分析
- 夏普比率評估

### 🎯 下一步
- 實作資料預處理模組
- 建立技術指標計算功能
- 準備 PatchTST 模型所需的特徵工程
- 加入更多技術分析指標 (RSI, MACD, 布林帶等)